# 🏭 Synthetic Production Data Generator + Scheduler
### 100% open-source — unique names every run, no API needed

| Section | What happens |
|---|---|
| **1–7** | Master data: Machines, Products, Process Steps (DGS approach) |
| **8** | ✏️ Schedule configuration |
| **9** | Schedule generator (shift-aware, no machine overlap, step sequencing) |
| **10** | Gantt chart — interactive timeline view |
| **11** | Schedule export |


## 1. Imports

In [1]:
import json, random, time, re, string
import pandas as pd
from datetime import datetime
from itertools import product as iterproduct
from IPython.display import display, JSON as DJSON

# ── Auto-seed from current time — different every single run ──
SEED = int(time.time())
random.seed(SEED)
print(f"🎲 Seed: {SEED}  — changes automatically every run")
print("   (To reproduce a specific dataset, set SEED manually in this cell)")


🎲 Seed: 1780331180  — changes automatically every run
   (To reproduce a specific dataset, set SEED manually in this cell)


## 2. ✏️ Configuration — Only Cell You Need to Edit


In [2]:
# ═══════════════════════════════════════════════════════════
#  ✏️  EDIT THESE → then  Kernel → Restart & Run All
# ═══════════════════════════════════════════════════════════

NUM_PRODUCTS      = 12    # Products to generate      (5–30)
NUM_MACHINES      = 8     # Machines to generate      (4–15)
STEPS_PER_PRODUCT = 5     # Average steps per product (3–8)

# ── Demand distribution ──────────────────────────────────
DEMAND_WEIGHTS = [0.2, 0.5, 0.3]   # low / mid / high probability
DEMAND_RANGES  = {
    "low":  (80,   400),
    "mid":  (400,  1500),
    "high": (1500, 4500),
}
BATCH_SIZES = [50, 100, 150, 200, 250]

# ═══════════════════════════════════════════════════════════
print(f"✅  {NUM_PRODUCTS} products | {NUM_MACHINES} machines | ~{STEPS_PER_PRODUCT} steps/product")


✅  12 products | 8 machines | ~5 steps/product


## 3. Grammar Vocabularies
These lists are combined algorithmically to produce thousands of unique names.  
Add or change words here to shift the naming style.


In [3]:
# ═══════════════════════════════════════════════════════════
#  MACHINE GRAMMAR
#  Brand + Model-prefix + Model-number → "Vortex ZX-740"
# ═══════════════════════════════════════════════════════════

MACHINE_BRANDS = [
    "Vortex", "Nexus", "Helios", "Axon", "Praxis",
    "Vektor", "Orbis", "Ferrum", "Stratos", "Callix",
    "Dynex", "Kronos", "Lumis", "Torq", "Zephyr",
    "Cryon", "Valdex", "Pinnex", "Solux", "Arcus",
]

MACHINE_MODEL_PREFIXES = ["ZX", "TX", "MX", "CX", "RX", "EX", "SX", "AX", "DX", "FX"]
MACHINE_MODEL_NUMBERS  = list(range(200, 1000, 20))   # 200, 220, 240 ... 980

STATION_OPERATIONS = [
    "Wire Cutting", "Cable Stripping", "Crimping", "Conductor Joining",
    "Terminal Insertion", "Connector Assembly", "Wire Rolling",
    "Tape Wrapping", "Tube Assembly", "Grommet Fitting",
    "Seal Insertion", "Cable Bundling", "Continuity Testing",
    "Pull-Force Testing", "Visual Inspection",
]

MOLDING_MACHINE_BRANDS = [
    "Engel", "Battenfeld", "Demag", "Wittmann", "Haitian",
    "Chen Hsong", "Toyo", "Toshiba", "Niigata", "Sumitomo",
]
MOLDING_MODEL_PREFIXES = ["ST", "XT", "GT", "HT", "ET", "MT"]
MOLDING_MODEL_NUMBERS  = list(range(250, 800, 25))

# ═══════════════════════════════════════════════════════════
#  PRODUCT GRAMMAR
# ═══════════════════════════════════════════════════════════

WIRE_CONFIGS = [
    ("Single Wire",    "single",  1),
    ("2-Wire Jacket",  "jacket",  2),
    ("3-Wire Jacket",  "jacket",  3),
    ("4-Wire Jacket",  "jacket",  4),
    ("5-Wire Jacket",  "jacket",  5),
    ("6-Wire Jacket",  "jacket",  6),
    ("8-Wire Jacket",  "jacket",  8),
    ("10-Wire Jacket", "jacket", 10),
    ("Twisted Pair",   "twisted", 2),
    ("Twisted Quad",   "twisted", 4),
    ("Shielded 2-Wire","shielded",2),
    ("Shielded 4-Wire","shielded",4),
    ("Coaxial Cable",  "coaxial", 1),
]

CONNECTOR_FAMILIES = [
    ("DCC",   ["1x", "2x", "3x"]),
    ("MQS",   ["1x", "2x"]),
    ("JPT",   ["1x", "2x", "3x"]),
    ("HSD",   ["1x", "2x"]),
    ("FAKRA", ["1x", "2x"]),
    ("HVL",   ["1x"]),
    ("AMP",   ["1x", "2x", "3x"]),
    ("RAST",  ["1x", "2x"]),
    ("Micro-Fit", ["1x", "2x"]),
    ("Mini-Fit",  ["1x", "2x", "3x"]),
]

# Part code formats — each is a lambda that returns a random code
PART_CODE_FORMATS = [
    lambda: f"{random.choice(string.ascii_uppercase)}{random.randint(1,9)}{random.randint(1000,9999)}",
    lambda: f"{random.randint(10,99)}{random.choice('ABCDEFGHJKLMNPQRSTVWXYZ')}{random.randint(100,999)}",
    lambda: f"{''.join(random.choices(string.ascii_uppercase,k=2))}{random.randint(10000,99999)}",
    lambda: f"{random.choice(string.ascii_uppercase)}{random.randint(10,99)}-{random.randint(100,999)}",
    lambda: f"{random.randint(100,999)}-{random.choice(string.ascii_uppercase)}{random.randint(10,99)}",
    lambda: f"{''.join(random.choices(string.digits,k=3))}{''.join(random.choices(string.ascii_uppercase,k=2))}{random.randint(10,99)}",
]

VARIANT_SUFFIXES = [""] * 3 + list("ABCDEFGHJKMNPRST")  # blank weighted 3x

SUBVARIANTS = [
    "AA/AB/AC", "AD/AE/AF", "BA/BB/BC", "CA/CB/CC",
    "DA/DB", "EA/EB/EC", "FA/FB", "X1/X2/X3",
]

# ═══════════════════════════════════════════════════════════
#  PROCESS STEP GRAMMAR
#  operation + qualifier + spec → "Stripping & Crimping 4-Wire, 120mm"
# ═══════════════════════════════════════════════════════════

STEP_OPERATIONS = {
    "cut_strip": [
        "Cutting & Stripping",
        "Precision Cutting",
        "Stripping & End-Preparation",
        "Jacket Removal & Stripping",
        "Wire Cutting & Separation",
        "Multi-Wire Stripping",
        "Conductor Exposure",
    ],
    "crimp": [
        "Terminal Crimping",
        "Crimping & Sleeve Insertion",
        "Contact Crimping",
        "End-Crimp Application",
        "Wire Crimping & Assembly",
        "Seal Crimping",
        "Ferrule Crimping",
    ],
    "assembly": [
        "Connector Housing Assembly",
        "Terminal Insertion",
        "Connector Body Assembly",
        "Pin Insertion & Lock",
        "Connector Mating & Latching",
        "Seal & Connector Assembly",
        "Manual Connector Build",
        "Sub-Assembly Integration",
    ],
    "tube_grommet": [
        "Corrugated Tube Fitting",
        "PUR Tube Assembly",
        "Grommet Insertion",
        "Tube & Grommet Sub-Assembly",
        "Protective Sleeve Fitting",
        "Conduit Assembly",
        "Rubber Grommet Seating",
    ],
    "wrap_tape": [
        "Wire Pair Rolling & Taping",
        "Cable Bundling & Taping",
        "Spiral Wrap Application",
        "PVC Tape Wrapping",
        "Protective Taping",
        "Harness Taping",
        "Cloth Tape Application",
    ],
    "mold": [
        "Overmolding",
        "Injection Overmolding",
        "Connector Overmolding",
        "Strain-Relief Molding",
        "Encapsulation Molding",
        "Insert Molding",
    ],
    "test": [
        "Electrical Continuity Test",
        "Pull-Force Verification",
        "Visual Quality Inspection",
        "HV Withstand Test",
        "Seal Integrity Check",
    ],
}

STEP_QUALIFIERS = {
    "cut_strip": [
        "{n}-Wire Jacket, {l}mm",
        "Jacket Cable {n}-Wire, Strip Length {s}mm",
        "Single Conductors, {l}mm Cut Length",
        "{n}-Core Cable, {l}mm",
        "Twisted {n}-Wire, Strip {s}mm / Cut {l}mm",
    ],
    "crimp": [
        "{connector} Contact, Wire Gauge {g} AWG",
        "{connector} Terminal, Crimp Force {f}N",
        "Tin-Plated Contact, {g} AWG",
        "{connector} Seal Crimp, {g} AWG",
        "Double Crimp — Insulation + Conductor",
    ],
    "assembly": [
        "{connector} Housing, {p}-Pin",
        "{connector} Body, {p}-Way, Colour {col}",
        "{connector} Connector, CPA {cpa}",
        "Coding {coding} — {orient}",
        "{p}-Position {connector}, Locking Clip",
    ],
    "tube_grommet": [
        "{td}mm × {tw}mm Tube, {l}mm Length",
        "ID {td}mm PUR Tube, {l}–{l2}mm",
        "Rubber Grommet {td}mm, {l}mm Cable",
        "Corrugated Tube Ø{td}mm, {l}mm",
        "Split Tube {td}mm, {l}mm Section",
    ],
    "wrap_tape": [
        "Wire Pairs, {l}mm Overlap",
        "Full Harness, {t}mm Tape Width",
        "Spiral 50% Overlap, {l}mm Section",
        "{t}mm PVC Tape, {l}mm Bundle Length",
        "Cloth Tape, Double-Wrap {l}mm",
    ],
    "mold": [
        "{orient} Connector, {part}, {coding}, {cpa}",
        "{orient} Entry, {coding}, {cpa}",
        "Straight-Exit, {part}, {cpa}",
        "{orient} Bend, {coding}, CPA Clip {cpa}",
    ],
    "test": [
        "All Circuits, {v}V Continuity",
        "Crimp Pull-Force ≥{f}N, Sample {pct}%",
        "Visual — Seal & Terminal Seating",
        "{v}V HV Isolation, 1s Dwell",
    ],
}

# Filler values for qualifiers
STEP_FILLERS = {
    "n": [2, 3, 4, 5, 6, 7, 8, 10],
    "l": [80, 100, 110, 120, 150, 175, 200, 250, 300, 400, 500],
    "l2": [200, 250, 300, 400, 500, 600],
    "s": [5, 6, 7, 8, 10, 12, 15],
    "g": ["0.35", "0.5", "0.75", "1.0", "1.5", "2.5"],
    "f": [30, 40, 50, 60, 80, 100],
    "v": [12, 24, 48, 60],
    "p": [2, 3, 4, 6, 8, 12],
    "t": [9, 15, 19, 25],
    "td": ["3.5", "5.0", "6.0", "7.0", "8.0"],
    "tw": ["1.25", "1.35", "1.50", "1.75"],
    "col": ["Black", "Grey", "White", "Natural"],
    "cpa": ["With CPA", "No CPA"],
    "coding": ["Cod-A Black", "Cod-B White", "Cod-C Blue", "Cod-D Grey"],
    "orient": ["180° Straight", "90° Bottom", "90° Right", "45° Angled"],
    "part": ["85E-973-752", "3Q0-973-752", "4P0-973-752", "9J1-973-752", "95C-973-752"],
    "pct": [5, 10, 20],
    "connector": [],  # filled dynamically per product
}

print("✅ Grammar vocabularies loaded")
print(f"   Machine brand combinations: {len(MACHINE_BRANDS) * len(MACHINE_MODEL_PREFIXES) * len(MACHINE_MODEL_NUMBERS):,}")
print(f"   Wire × Connector combinations: {len(WIRE_CONFIGS) * sum(len(v) for _,v in CONNECTOR_FAMILIES):,}")
print(f"   Step operation × qualifier combinations: {sum(len(v) for v in STEP_OPERATIONS.values()) * sum(len(v) for v in STEP_QUALIFIERS.values()):,}")


✅ Grammar vocabularies loaded
   Machine brand combinations: 8,000
   Wire × Connector combinations: 299
   Step operation × qualifier combinations: 1,551


## 4. Generator Functions

In [4]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def rc(lst):
    """Random choice shorthand."""
    return random.choice(lst)

def fill_qualifier(template, connector="DCC"):
    """Fill {placeholders} in a step qualifier template."""
    STEP_FILLERS["connector"] = [connector]
    t = template
    for key, values in STEP_FILLERS.items():
        if f"{{{key}}}" in t and values:
            t = t.replace(f"{{{key}}}", str(rc(values)))
    # Fill any remaining with a sensible default
    t = re.sub(r"\{[^}]+\}", "", t)
    return t.strip().strip(",").strip()

def rand_cycle(low, high):
    return round(random.uniform(low, high), 2)

print("✅ Helpers ready")


✅ Helpers ready


In [5]:
# ── Machine Generator ────────────────────────────────────────────────────────

def generate_machines(n):
    """
    Builds n machines with fresh combinatorial names each run.
    Splits into 3 role buckets:
      - cutting/crimping pair machines  (~30%)
      - assembly stations               (~35%)
      - overmolding machines            (~35%)
    """
    machines = []
    used_names = set()

    def unique_name(candidates):
        for _ in range(50):
            name = rc(candidates)
            if name not in used_names:
                used_names.add(name)
                return name
        return candidates[0]  # fallback

    n_molding  = max(1, round(n * 0.30))
    n_cutting  = max(1, round(n * 0.30))
    n_stations = n - n_molding - n_cutting

    # Cutting / crimping pair machines
    for _ in range(n_cutting):
        b1, b2 = random.sample(MACHINE_BRANDS, 2)
        m1 = f"{rc(MACHINE_MODEL_PREFIXES)}-{rc(MACHINE_MODEL_NUMBERS)}"
        m2 = f"{rc(MACHINE_MODEL_PREFIXES)}-{rc(MACHINE_MODEL_NUMBERS)}"
        name = f"{b1} {m1} / {b2} {m2}"
        if name in used_names:
            m1 = f"{rc(MACHINE_MODEL_PREFIXES)}-{rc(MACHINE_MODEL_NUMBERS)}"
            name = f"{b1} {m1} / {b2} {m2}"
        used_names.add(name)
        machines.append({"name": name, "available_hours_per_day": 24.0, "_role": "cutting"})

    # Assembly stations
    ops = random.sample(STATION_OPERATIONS, min(n_stations, len(STATION_OPERATIONS)))
    if len(ops) < n_stations:
        ops += random.choices(STATION_OPERATIONS, k=n_stations - len(ops))
    for op in ops[:n_stations]:
        name = f"{op} Station"
        if name in used_names:
            name = f"{op} & QC Station"
        used_names.add(name)
        machines.append({"name": name, "available_hours_per_day": 16.0, "_role": "assembly"})

    # Overmolding / injection machines (grouped)
    brand = rc(MOLDING_MACHINE_BRANDS)
    model = f"{rc(MOLDING_MODEL_PREFIXES)}{rc(MOLDING_MODEL_NUMBERS)}"
    per_group = max(4, round(12 / max(n_molding, 1)))
    start = 1
    for _ in range(n_molding):
        end = start + per_group - 1
        name = f"{brand} {model} Machine {start}-{end}"
        used_names.add(name)
        machines.append({"name": name, "available_hours_per_day": 24.0, "_role": "molding"})
        start = end + 1
        # New brand/model for variety if more than one group
        brand = rc(MOLDING_MACHINE_BRANDS)
        model = f"{rc(MOLDING_MODEL_PREFIXES)}{rc(MOLDING_MODEL_NUMBERS)}"

    return machines


print("✅ Machine generator ready")


✅ Machine generator ready


In [6]:
# ── Product Generator ────────────────────────────────────────────────────────

def generate_products(n):
    """
    Generates n products with unique descriptions.
    Part codes are generated from rotating format lambdas so they
    look different every run (not always '9Y42xx' style).
    """
    products = []
    used_descs = set()

    for i in range(1, n + 1):
        # Wire config
        wire_label, wire_category, wire_count = rc(WIRE_CONFIGS)

        # Connector
        conn_family, conn_multipliers = rc(CONNECTOR_FAMILIES)
        conn_multiplier = rc(conn_multipliers)
        connector_str = f"{conn_multiplier}{conn_family}"
        dcc_type_map = {"1x": "Single", "2x": "Dual", "3x": "Triple"}
        dcc_type = f"{dcc_type_map.get(conn_multiplier, conn_multiplier)} {conn_family}"

        # Part code — pick a fresh format each time
        part_code_fn = rc(PART_CODE_FORMATS)
        part_code = part_code_fn()
        suffix = rc(VARIANT_SUFFIXES)

        # Subvariant for twisted/shielded
        subvariant = ""
        if wire_category in ("twisted", "shielded") and random.random() > 0.4:
            subvariant = f" {rc(SUBVARIANTS)}"

        description = f"{wire_label} {connector_str} Module {part_code}{suffix}{subvariant}"

        # Avoid duplicate descriptions
        if description in used_descs:
            part_code = part_code_fn()
            description = f"{wire_label} {connector_str} Module {part_code}{suffix}{subvariant}"
        used_descs.add(description)

        # Demand
        tier   = random.choices(["low", "mid", "high"], weights=DEMAND_WEIGHTS)[0]
        demand = random.randint(*DEMAND_RANGES[tier])
        batch  = rc(BATCH_SIZES)

        products.append({
            "item":         i,
            "sap_tn":       f"TN-{random.randint(100000, 999999)}",
            "sap_pl":       f"PL-{random.randint(1000, 9999)}" if random.random() > 0.25 else None,
            "dcc_type":     dcc_type,
            "description":  description,
            "demand_2024":  demand,
            "batch_size":   batch,
            "num_batches":  max(1, round(demand / batch)),
            # internal — stripped on export
            "_wire_category": wire_category,
            "_connector":     conn_family,
            "_wire_count":    wire_count,
        })

    return products


print("✅ Product generator ready")


✅ Product generator ready


In [7]:
# ── Process Step Generator ───────────────────────────────────────────────────

def build_step(op_category, product_connector, machine_name, step_num, product_item,
               cycle_range, workers):
    """Build one process step dict."""
    operation  = rc(STEP_OPERATIONS[op_category])
    qualifier_tmpl = rc(STEP_QUALIFIERS[op_category])
    qualifier  = fill_qualifier(qualifier_tmpl, connector=product_connector)
    step_name  = f"{operation} — {qualifier}" if qualifier else operation

    return {
        "product_item":       product_item,
        "step_number":        step_num,
        "machine_name":       machine_name,
        "step_name":          step_name,
        "cycle_time_seconds": rand_cycle(*cycle_range),
        "workers_required":   workers,
    }


def generate_process_steps(products, machines, avg_steps):
    """
    Assigns process steps to each product following factory flow:

        1. Cut & Strip       → cutting machine
        2. Crimp             → cutting machine (same or different)
        3. Tube / Grommet    → assembly station  [optional ~60%]
        4. Connector Assy    → assembly station
        5. Wrap / Tape       → assembly station  [optional ~40%]
        6. Overmolding       → molding machine   [always last]
        +  Test              → assembly station  [optional ~50%]

    Steps per product vary ±1 around avg_steps.
    """
    # Index machines by role
    cutting_m  = [m["name"] for m in machines if m.get("_role") == "cutting"]
    assembly_m = [m["name"] for m in machines if m.get("_role") == "assembly"]
    molding_m  = [m["name"] for m in machines if m.get("_role") == "molding"]

    # Fallbacks
    if not cutting_m:  cutting_m  = [machines[0]["name"]]
    if not assembly_m: assembly_m = [machines[1]["name"]]
    if not molding_m:  molding_m  = [machines[-1]["name"]]

    all_steps = []

    for product in products:
        item       = product["item"]
        connector  = product["_connector"]
        category   = product["_wire_category"]
        steps      = []
        step_num   = 1

        def add(op_cat, machine_pool, cycle_range, workers=0.5):
            nonlocal step_num
            steps.append(build_step(
                op_cat, connector, rc(machine_pool),
                step_num, item, cycle_range, workers
            ))
            step_num += 1

        # 1. Cut & Strip (always)
        add("cut_strip", cutting_m, (5.5, 9.5), workers=0.5)

        # 2. Crimp (always)
        add("crimp", cutting_m, (5.0, 9.0), workers=0.5)

        # 3. Tube / Grommet (jacket/shielded ~65%, twisted ~25%)
        tube_prob = 0.65 if category in ("jacket","shielded") else 0.25
        if random.random() < tube_prob:
            add("tube_grommet", assembly_m, (9.0, 20.0), workers=1.0)

        # 4. Connector Assembly (always)
        add("assembly", assembly_m, (7.0, 14.0), workers=1.0)

        # 5. Wrap / Tape (twisted ~80%, coaxial ~60%, others ~30%)
        wrap_probs = {"twisted": 0.80, "coaxial": 0.60, "jacket": 0.30, "shielded": 0.45, "single": 0.20}
        if random.random() < wrap_probs.get(category, 0.30):
            add("wrap_tape", assembly_m, (7.5, 13.0), workers=0.5)

        # 6. Optional extra assembly step to hit avg_steps target
        current = len(steps)
        target  = avg_steps + random.randint(-1, 1)
        if current < target - 1:   # room for one more before molding
            add("assembly", assembly_m, (8.0, 15.0), workers=1.0)

        # 7. Test step (50% chance)
        if random.random() < 0.50:
            add("test", assembly_m, (4.0, 8.0), workers=0.5)

        # 8. Overmolding — always last
        add("mold", molding_m, (13.0, 22.0), workers=1.0)

        all_steps.extend(steps)

    return all_steps


print("✅ Process step generator ready")


✅ Process step generator ready


## 5. Generate Data

In [8]:
# Re-seed here so Restart & Run All always gives fresh output
random.seed(int(time.time()))

print("⏳ Generating...")
machines      = generate_machines(NUM_MACHINES)
products      = generate_products(NUM_PRODUCTS)
process_steps = generate_process_steps(products, machines, STEPS_PER_PRODUCT)

# Build DataFrames for display/export
df_machines = pd.DataFrame([{k:v for k,v in m.items() if not k.startswith("_")} for m in machines])
df_products = pd.DataFrame([{k:v for k,v in p.items() if not k.startswith("_")} for p in products])
df_steps    = pd.DataFrame(process_steps)

print()
print("═" * 55)
print(f"  ✅ Machines:       {len(machines)}")
print(f"  ✅ Products:       {len(products)}")
print(f"  ✅ Process Steps:  {len(process_steps)}  (avg {len(process_steps)/len(products):.1f}/product)")
print("═" * 55)


⏳ Generating...

═══════════════════════════════════════════════════════
  ✅ Machines:       8
  ✅ Products:       12
  ✅ Process Steps:  68  (avg 5.7/product)
═══════════════════════════════════════════════════════


## 6. Explore & Validate

In [9]:
print("=== MACHINES ===")
display(df_machines)


=== MACHINES ===


,name,available_hours_per_day
0,Lumis ZX-880 / Stratos CX-680,24.0
1,Stratos EX-200 / Vortex DX-700,24.0
2,Grommet Fitting Station,16.0
3,Connector Assembly Station,16.0
4,Continuity Testing Station,16.0
5,Wire Rolling Station,16.0
6,Niigata XT525 Machine 1-6,24.0
7,Engel HT775 Machine 7-12,24.0


In [10]:
print("=== PRODUCTS ===")
display(df_products[["item","description","dcc_type","demand_2024","batch_size","num_batches"]])


=== PRODUCTS ===


,item,description,dcc_type,demand_2024,batch_size,num_batches
0,1,4-Wire Jacket 1xMQS Module 16K915H,Single MQS,1201,200,6
1,2,Coaxial Cable 1xHVL Module H22-105F,Single HVL,991,100,10
2,3,3-Wire Jacket 2xFAKRA Module 365DG87,Dual FAKRA,911,250,4
3,4,Single Wire 3xMini-Fit Module U20-640B,Triple Mini-Fit,4019,50,80
4,5,Shielded 4-Wire 1xHSD Module VQ29592A BA/BB/BC,Single HSD,2014,200,10
5,6,8-Wire Jacket 3xMini-Fit Module 84M305F,Triple Mini-Fit,3825,150,26
6,7,Twisted Quad 2xMini-Fit Module 14C821P DA/DB,Dual Mini-Fit,155,100,2
7,8,Coaxial Cable 1xJPT Module 446-K20S,Single JPT,2088,250,8
8,9,Single Wire 1xFAKRA Module 674MG43,Single FAKRA,411,150,3
9,10,Twisted Quad 1xHSD Module 63L878P BA/BB/BC,Single HSD,408,250,2


In [11]:
print("=== PROCESS STEPS — first 20 rows ===")
display(df_steps.head(20))


=== PROCESS STEPS — first 20 rows ===


,product_item,step_number,machine_name,step_name,cycle_time_seconds,workers_required
0,1,1,Stratos EX-200 / Vortex DX-700,"Precision Cutting — Jacket Cable 10-Wire, Stri...",9.00,0.5
1,1,2,Stratos EX-200 / Vortex DX-700,"End-Crimp Application — MQS Contact, Wire Gaug...",8.35,0.5
2,1,3,Connector Assembly Station,"Conduit Assembly — Split Tube 6.0mm, 175mm Sec...",9.94,1.0
3,1,4,Connector Assembly Station,"Manual Connector Build — 12-Position MQS, Lock...",13.31,1.0
4,1,5,Continuity Testing Station,Pull-Force Verification — Visual — Seal & Term...,6.38,0.5
5,1,6,Niigata XT525 Machine 1-6,"Encapsulation Molding — Straight-Exit, 95C-973...",21.89,1.0
6,2,1,Lumis ZX-880 / Stratos CX-680,"Precision Cutting — Twisted 8-Wire, Strip 15mm...",8.90,0.5
7,2,2,Lumis ZX-880 / Stratos CX-680,Ferrule Crimping — Double Crimp — Insulation +...,7.45,0.5
8,2,3,Wire Rolling Station,"Grommet Insertion — 3.5mm × 1.35mm Tube, 500mm...",14.43,1.0
9,2,4,Grommet Fitting Station,"Connector Mating & Latching — HVL Connector, C...",10.01,1.0


In [12]:
print("=== STEP COUNT PER PRODUCT ===")
display(df_steps.groupby("product_item").size().rename("step_count").to_frame())


=== STEP COUNT PER PRODUCT ===


,step_count
product_item,
1,6
2,5
3,6
4,5
5,6
6,7
7,5
8,6
9,5


In [13]:
print("=== MACHINE WORKLOAD ===")
display(df_steps.groupby("machine_name").size().rename("steps_assigned")
        .sort_values(ascending=False).to_frame())


=== MACHINE WORKLOAD ===


,steps_assigned
machine_name,
Stratos EX-200 / Vortex DX-700,12
Lumis ZX-880 / Stratos CX-680,12
Grommet Fitting Station,11
Connector Assembly Station,9
Niigata XT525 Machine 1-6,8
Wire Rolling Station,8
Engel HT775 Machine 7-12,4
Continuity Testing Station,4


In [14]:
print("=== ALL UNIQUE STEP NAMES ===")
for s in sorted(df_steps["step_name"].unique()):
    print(f"  • {s}")


=== ALL UNIQUE STEP NAMES ===
  • Cable Bundling & Taping — Cloth Tape, Double-Wrap 175mm
  • Conduit Assembly — Split Tube 6.0mm, 175mm Section
  • Connector Body Assembly — Coding Cod-A Black — 90° Right
  • Connector Housing Assembly — 2-Position Mini-Fit, Locking Clip
  • Connector Housing Assembly — Coding Cod-B White — 90° Bottom
  • Connector Mating & Latching — HVL Connector, CPA With CPA
  • Connector Overmolding — Straight-Exit, 4P0-973-752, No CPA
  • Contact Crimping — Double Crimp — Insulation + Conductor
  • Contact Crimping — Mini-Fit Terminal, Crimp Force 60N
  • Crimping & Sleeve Insertion — Double Crimp — Insulation + Conductor
  • Electrical Continuity Test — Visual — Seal & Terminal Seating
  • Encapsulation Molding — 90° Bottom Bend, Cod-A Black, CPA Clip No CPA
  • Encapsulation Molding — Straight-Exit, 95C-973-752, With CPA
  • End-Crimp Application — Double Crimp — Insulation + Conductor
  • End-Crimp Application — MQS Contact, Wire Gauge 0.5 AWG
  • Ferrule Cri

In [15]:
print("=== FULL PRODUCT → STEP TRACE ===")
trace = df_steps.merge(df_products[["item","description"]], left_on="product_item", right_on="item")
display(trace[["item","description","step_number","step_name","machine_name","cycle_time_seconds"]])


=== FULL PRODUCT → STEP TRACE ===


,item,description,step_number,step_name,machine_name,cycle_time_seconds
0,1,4-Wire Jacket 1xMQS Module 16K915H,1,"Precision Cutting — Jacket Cable 10-Wire, Stri...",Stratos EX-200 / Vortex DX-700,9.00
1,1,4-Wire Jacket 1xMQS Module 16K915H,2,"End-Crimp Application — MQS Contact, Wire Gaug...",Stratos EX-200 / Vortex DX-700,8.35
2,1,4-Wire Jacket 1xMQS Module 16K915H,3,"Conduit Assembly — Split Tube 6.0mm, 175mm Sec...",Connector Assembly Station,9.94
3,1,4-Wire Jacket 1xMQS Module 16K915H,4,"Manual Connector Build — 12-Position MQS, Lock...",Connector Assembly Station,13.31
4,1,4-Wire Jacket 1xMQS Module 16K915H,5,Pull-Force Verification — Visual — Seal & Term...,Continuity Testing Station,6.38
...,...,...,...,...,...,...
63,12,4-Wire Jacket 2xRAST Module M43701J,1,"Precision Cutting — 4-Core Cable, 400mm",Lumis ZX-880 / Stratos CX-680,6.47
64,12,4-Wire Jacket 2xRAST Module M43701J,2,"Ferrule Crimping — RAST Contact, Wire Gauge 2....",Lumis ZX-880 / Stratos CX-680,8.70
65,12,4-Wire Jacket 2xRAST Module M43701J,3,"Seal & Connector Assembly — 12-Position RAST, ...",Connector Assembly Station,11.68
66,12,4-Wire Jacket 2xRAST Module M43701J,4,"Sub-Assembly Integration — RAST Housing, 3-Pin",Wire Rolling Station,9.41


## 7. Export

In [16]:
def clean(lst):
    return [{k:v for k,v in d.items() if not k.startswith("_")} for d in lst]

# JSON
output = {
    "generated_at": datetime.now().isoformat(),
    "seed": SEED,
    "machines":      clean(machines),
    "products":      clean(products),
    "process_steps": process_steps,
}
with open("synthetic_production_data.json","w") as f:
    json.dump(output, f, indent=2)

# CSV
df_machines.to_csv("machines.csv", index=False)
df_products.to_csv("products.csv", index=False)
df_steps.to_csv("process_steps.csv", index=False)

print("Machines")
display(df_machines)
print("Products")
display(df_products)
print("Process Steps")
display(df_steps)

print("✅ Exported:")
print("   synthetic_production_data.json")
print(f"   machines.csv       ({len(machines)} rows)")
print(f"   products.csv       ({len(products)} rows)")
print(f"   process_steps.csv  ({len(process_steps)} rows)")


Machines


,name,available_hours_per_day
0,Lumis ZX-880 / Stratos CX-680,24.0
1,Stratos EX-200 / Vortex DX-700,24.0
2,Grommet Fitting Station,16.0
3,Connector Assembly Station,16.0
4,Continuity Testing Station,16.0
5,Wire Rolling Station,16.0
6,Niigata XT525 Machine 1-6,24.0
7,Engel HT775 Machine 7-12,24.0


Products


,item,sap_tn,sap_pl,dcc_type,description,demand_2024,batch_size,num_batches
0,1,TN-114118,None,Single MQS,4-Wire Jacket 1xMQS Module 16K915H,1201,200,6
1,2,TN-872498,None,Single HVL,Coaxial Cable 1xHVL Module H22-105F,991,100,10
2,3,TN-102716,PL-8962,Dual FAKRA,3-Wire Jacket 2xFAKRA Module 365DG87,911,250,4
3,4,TN-555384,PL-8873,Triple Mini-Fit,Single Wire 3xMini-Fit Module U20-640B,4019,50,80
4,5,TN-230378,PL-2682,Single HSD,Shielded 4-Wire 1xHSD Module VQ29592A BA/BB/BC,2014,200,10
5,6,TN-674567,PL-3371,Triple Mini-Fit,8-Wire Jacket 3xMini-Fit Module 84M305F,3825,150,26
6,7,TN-219028,PL-7904,Dual Mini-Fit,Twisted Quad 2xMini-Fit Module 14C821P DA/DB,155,100,2
7,8,TN-423499,PL-5674,Single JPT,Coaxial Cable 1xJPT Module 446-K20S,2088,250,8
8,9,TN-926465,PL-7522,Single FAKRA,Single Wire 1xFAKRA Module 674MG43,411,150,3
9,10,TN-122169,PL-1730,Single HSD,Twisted Quad 1xHSD Module 63L878P BA/BB/BC,408,250,2


Process Steps


,product_item,step_number,machine_name,step_name,cycle_time_seconds,workers_required
0,1,1,Stratos EX-200 / Vortex DX-700,"Precision Cutting — Jacket Cable 10-Wire, Stri...",9.00,0.5
1,1,2,Stratos EX-200 / Vortex DX-700,"End-Crimp Application — MQS Contact, Wire Gaug...",8.35,0.5
2,1,3,Connector Assembly Station,"Conduit Assembly — Split Tube 6.0mm, 175mm Sec...",9.94,1.0
3,1,4,Connector Assembly Station,"Manual Connector Build — 12-Position MQS, Lock...",13.31,1.0
4,1,5,Continuity Testing Station,Pull-Force Verification — Visual — Seal & Term...,6.38,0.5
...,...,...,...,...,...,...
63,12,1,Lumis ZX-880 / Stratos CX-680,"Precision Cutting — 4-Core Cable, 400mm",6.47,0.5
64,12,2,Lumis ZX-880 / Stratos CX-680,"Ferrule Crimping — RAST Contact, Wire Gauge 2....",8.70,0.5
65,12,3,Connector Assembly Station,"Seal & Connector Assembly — 12-Position RAST, ...",11.68,1.0
66,12,4,Wire Rolling Station,"Sub-Assembly Integration — RAST Housing, 3-Pin",9.41,1.0


✅ Exported:
   synthetic_production_data.json
   machines.csv       (8 rows)
   products.csv       (12 rows)
   process_steps.csv  (68 rows)


In [17]:
# Django fixtures (optional — replace 'yourapp' with your app label)
APP = "yourapp"
fixtures = []
for m in clean(machines):
    fixtures.append({"model": f"{APP}.machine", "fields": m})
for p in clean(products):
    fixtures.append({"model": f"{APP}.product", "pk": p["item"],
                     "fields": {k:v for k,v in p.items() if k != "item"}})
for s in process_steps:
    fixtures.append({"model": f"{APP}.processstep", "fields": s})

with open("fixtures.json","w") as f:
    json.dump(fixtures, f, indent=2)

print(f"✅ Django fixtures → fixtures.json  ({len(fixtures)} entries)")
print(f"   Run: python manage.py loaddata fixtures.json")


✅ Django fixtures → fixtures.json  (88 entries)
   Run: python manage.py loaddata fixtures.json


## 8. Raw JSON Preview — First Product

In [18]:
first = clean(products)[0]
first_steps = [s for s in process_steps if s["product_item"] == first["item"]]
display(DJSON({"product": first, "process_steps": first_steps}))


<IPython.core.display.JSON object>

---
## 8. ✏️ Schedule Configuration
Set the scheduling window and shift pattern. Then run sections 9–10.


In [19]:
from datetime import datetime, timedelta

# ═══════════════════════════════════════════════════════════
#  ✏️  EDIT THESE
# ═══════════════════════════════════════════════════════════

SCHEDULE_START   = datetime(2024, 1, 15, 6, 0)   # First shift start
SCHEDULE_DAYS    = 5                               # How many days to schedule
NUM_BATCHES_CAP  = 3                               # Max batches per product to schedule (keep small for clarity)

# Shift windows (start_hour, end_hour) — 24h clock
# Default: 3 shifts × 8h
SHIFTS = [
    (6,  14),   # Morning shift
    (14, 22),   # Afternoon shift
    (22, 30),   # Night shift  (30 = next-day 06:00)
]

# ═══════════════════════════════════════════════════════════
print(f"✅ Schedule: {SCHEDULE_DAYS} days from {SCHEDULE_START.strftime('%Y-%m-%d %H:%M')}")
print(f"   Shifts: {SHIFTS}")
print(f"   Max batches/product: {NUM_BATCHES_CAP}")


✅ Schedule: 5 days from 2024-01-15 06:00
   Shifts: [(6, 14), (14, 22), (22, 30)]
   Max batches/product: 3


## 9. Schedule Generator
**Protocols enforced:**
- ✅ No machine overlap — machine freed before next job starts
- ✅ Step sequencing — Step N+1 starts only after Step N ends (same batch)
- ✅ Shift boundaries — jobs start/end within valid shift windows
- ✅ Duration consistency — `duration_hours = end_time - start_time`
- ✅ Batch coherence — batch_id carries consistent product/batch_size/batch_num
- ✅ Multi-product sharing — machines pooled across all products


In [20]:
# ── Shift utilities ──────────────────────────────────────────────────────────

def get_shift_windows(base_date, shifts, num_days):
    """
    Returns list of (shift_start, shift_end) datetime pairs
    for num_days starting from base_date.
    """
    windows = []
    for day_offset in range(num_days + 1):          # +1 for night shift overflow
        day = base_date.replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=day_offset)
        for (sh, eh) in shifts:
            s = day + timedelta(hours=sh)
            e = day + timedelta(hours=eh)            # eh>24 handled by timedelta
            windows.append((s, e))
    windows.sort()
    return windows


def find_slot(machine_free_at, earliest_start, duration_h, shift_windows):
    """
    Find the earliest valid slot for a job respecting:
    - machine must be free (>= machine_free_at)
    - start >= earliest_start (step sequencing)
    - slot must fit entirely within one shift window
    Returns (start, end) datetimes.
    """
    candidate = max(machine_free_at, earliest_start)
    duration  = timedelta(hours=duration_h)

    for (sw_start, sw_end) in shift_windows:
        if sw_end <= candidate:
            continue                              # window already passed
        # Snap candidate to shift start if it falls before this window
        actual_start = max(candidate, sw_start)
        actual_end   = actual_start + duration
        if actual_end <= sw_end:                 # fits in this window
            return actual_start, actual_end
        # Job doesn't fit in this window — try next
    raise RuntimeError("Could not find a valid slot — extend SCHEDULE_DAYS")


# ── Main scheduler ───────────────────────────────────────────────────────────

def generate_schedule(products, machines, process_steps, schedule_start,
                      shifts, num_days, num_batches_cap):
    """
    Discrete-event scheduler using a machine availability dictionary.
    
    Algorithm:
      For each product → for each batch → for each step (in order):
        1. Compute duration from cycle_time_seconds × batch_size
        2. Find earliest slot on the required machine respecting:
               a) machine_free_at[machine]  (no overlap)
               b) batch_prev_step_end       (sequencing)
               c) shift window boundaries
        3. Book the slot → update machine_free_at
        4. Record the schedule entry
    """
    shift_windows = get_shift_windows(schedule_start, shifts, num_days)

    # machine availability tracker: machine_name → datetime when free
    machine_free_at = {m["name"]: schedule_start for m in machines}

    # Index steps by product
    steps_by_product = {}
    for s in process_steps:
        steps_by_product.setdefault(s["product_item"], []).append(s)
    for pid in steps_by_product:
        steps_by_product[pid].sort(key=lambda x: x["step_number"])

    schedule_rows = []

    for product in products:
        pid        = product["item"]
        batch_size = product["batch_size"]
        n_batches  = min(product["num_batches"], num_batches_cap)
        steps      = steps_by_product.get(pid, [])

        for batch_num in range(1, n_batches + 1):
            batch_id       = f"B-{pid:03d}-{batch_num:03d}"
            prev_step_end  = schedule_start        # sequencing anchor

            for step in steps:
                machine      = step["machine_name"]
                cycle_secs   = step["cycle_time_seconds"]

                # duration = cycle time per unit × batch size → convert to hours
                duration_h   = round((cycle_secs * batch_size) / 3600, 4)
                duration_h   = max(duration_h, 0.25)  # minimum 15 min slot

                start, end = find_slot(
                    machine_free_at=machine_free_at[machine],
                    earliest_start=prev_step_end,
                    duration_h=duration_h,
                    shift_windows=shift_windows,
                )

                # Update machine availability
                machine_free_at[machine] = end
                prev_step_end            = end

                schedule_rows.append({
                    "batch_id":       batch_id,
                    "batch_num":      batch_num,
                    "batch_size":     batch_size,
                    "product_item":   pid,
                    "product_desc":   product["description"],
                    "step_number":    step["step_number"],
                    "step_name":      step["step_name"],
                    "machine_name":   machine,
                    "start_time":     start,
                    "end_time":       end,
                    "duration_hours": round((end - start).total_seconds() / 3600, 4),
                })

    return schedule_rows


# ── Run ──────────────────────────────────────────────────────────────────────
print("⏳ Scheduling...")
schedule = generate_schedule(
    products, machines, process_steps,
    SCHEDULE_START, SHIFTS, SCHEDULE_DAYS, NUM_BATCHES_CAP
)
df_schedule = pd.DataFrame(schedule)

# ── Validation ───────────────────────────────────────────────────────────────
errors = []

# 1. Duration consistency
df_schedule["_calc_dur"] = (df_schedule["end_time"] - df_schedule["start_time"]).dt.total_seconds() / 3600
dur_mismatch = df_schedule[abs(df_schedule["_calc_dur"] - df_schedule["duration_hours"]) > 0.001]
if len(dur_mismatch): errors.append(f"Duration mismatch: {len(dur_mismatch)} rows")

# 2. Machine overlap check
for machine, grp in df_schedule.groupby("machine_name"):
    grp = grp.sort_values("start_time")
    for i in range(len(grp) - 1):
        if grp.iloc[i]["end_time"] > grp.iloc[i+1]["start_time"]:
            errors.append(f"Machine overlap: {machine}")
            break

# 3. Step sequencing check
for (pid, batch_num), grp in df_schedule.groupby(["product_item","batch_num"]):
    grp = grp.sort_values("step_number")
    for i in range(len(grp) - 1):
        if grp.iloc[i]["end_time"] > grp.iloc[i+1]["start_time"]:
            errors.append(f"Step sequence violation: product {pid} batch {batch_num}")
            break

df_schedule = df_schedule.drop(columns=["_calc_dur"])

print(f"\n✅ Schedule generated: {len(df_schedule)} entries")
if errors:
    print("⚠️  Validation issues:")
    for e in errors: print(f"   {e}")
else:
    print("✅ All validations passed:")
    print("   • No machine overlaps")
    print("   • Step sequencing respected")
    print("   • Duration consistency verified")

print(f"\n   Date range: {df_schedule['start_time'].min().strftime('%Y-%m-%d %H:%M')} → {df_schedule['end_time'].max().strftime('%Y-%m-%d %H:%M')}")
print(f"   Products scheduled: {df_schedule['product_item'].nunique()}")
print(f"   Machines used: {df_schedule['machine_name'].nunique()}")
print(f"   Batches: {df_schedule['batch_id'].nunique()}")


⏳ Scheduling...

✅ Schedule generated: 194 entries
✅ All validations passed:
   • No machine overlaps
   • Step sequencing respected
   • Duration consistency verified

   Date range: 2024-01-15 06:00 → 2024-01-16 10:15
   Products scheduled: 12
   Machines used: 8
   Batches: 34


In [21]:
# ── Schedule table preview ───────────────────────────────────────────────────
display(df_schedule[[
    "batch_id","product_desc","step_number","step_name",
    "machine_name","start_time","end_time","duration_hours"
]].head(20))


,batch_id,product_desc,step_number,step_name,machine_name,start_time,end_time,duration_hours
0,B-001-001,4-Wire Jacket 1xMQS Module 16K915H,1,"Precision Cutting — Jacket Cable 10-Wire, Stri...",Stratos EX-200 / Vortex DX-700,2024-01-15 06:00:00.000,2024-01-15 06:30:00.000,0.5000
1,B-001-001,4-Wire Jacket 1xMQS Module 16K915H,2,"End-Crimp Application — MQS Contact, Wire Gaug...",Stratos EX-200 / Vortex DX-700,2024-01-15 06:30:00.000,2024-01-15 06:57:50.040,0.4639
2,B-001-001,4-Wire Jacket 1xMQS Module 16K915H,3,"Conduit Assembly — Split Tube 6.0mm, 175mm Sec...",Connector Assembly Station,2024-01-15 06:57:50.040,2024-01-15 07:30:57.960,0.5522
3,B-001-001,4-Wire Jacket 1xMQS Module 16K915H,4,"Manual Connector Build — 12-Position MQS, Lock...",Connector Assembly Station,2024-01-15 07:30:57.960,2024-01-15 08:15:19.800,0.7394
4,B-001-001,4-Wire Jacket 1xMQS Module 16K915H,5,Pull-Force Verification — Visual — Seal & Term...,Continuity Testing Station,2024-01-15 08:15:19.800,2024-01-15 08:36:35.640,0.3544
5,B-001-001,4-Wire Jacket 1xMQS Module 16K915H,6,"Encapsulation Molding — Straight-Exit, 95C-973...",Niigata XT525 Machine 1-6,2024-01-15 08:36:35.640,2024-01-15 09:49:33.600,1.2161
6,B-001-002,4-Wire Jacket 1xMQS Module 16K915H,1,"Precision Cutting — Jacket Cable 10-Wire, Stri...",Stratos EX-200 / Vortex DX-700,2024-01-15 06:57:50.040,2024-01-15 07:27:50.040,0.5000
7,B-001-002,4-Wire Jacket 1xMQS Module 16K915H,2,"End-Crimp Application — MQS Contact, Wire Gaug...",Stratos EX-200 / Vortex DX-700,2024-01-15 07:27:50.040,2024-01-15 07:55:40.080,0.4639
8,B-001-002,4-Wire Jacket 1xMQS Module 16K915H,3,"Conduit Assembly — Split Tube 6.0mm, 175mm Sec...",Connector Assembly Station,2024-01-15 08:15:19.800,2024-01-15 08:48:27.720,0.5522
9,B-001-002,4-Wire Jacket 1xMQS Module 16K915H,4,"Manual Connector Build — 12-Position MQS, Lock...",Connector Assembly Station,2024-01-15 08:48:27.720,2024-01-15 09:32:49.560,0.7394


## 10. Gantt Chart — Timeline View

One reusable `plot_gantt()` function handles all views.  
**`product_id`** — an integer, or `"All"`  
**`batch_ids`** — a list like `["B-001-001", "B-001-002"]`, or `"All"`  
**`machine_names`** — a list of machine name strings, or `"All"`


### Define `plot_gantt()` — run once

In [22]:
import plotly.graph_objects as go

def plot_gantt(df_schedule, product_id="All", batch_ids="All", machine_names="All"):
    """
    Flexible Gantt chart for production schedule data.

    Parameters
    ----------
    df_schedule   : DataFrame  — full schedule from Section 9
    product_id    : int | "All"         — filter to one product or show all
    batch_ids     : list[str] | "All"   — filter to specific batches or show all
    machine_names : list[str] | "All"   — filter to specific machines or show all

    Colour logic
    ------------
    Single product  → bars coloured by batch
    All products    → bars coloured by product
    """
    df = df_schedule.copy()

    # ── Filters ──────────────────────────────────────────────────────────────
    if product_id != "All":
        df = df[df["product_item"] == product_id]
    if batch_ids != "All":
        df = df[df["batch_id"].isin(batch_ids)]
    if machine_names != "All":
        df = df[df["machine_name"].isin(machine_names)]

    if df.empty:
        print("⚠️  No data matches these filters.")
        print(f"   Available products : {sorted(df_schedule['product_item'].unique())}")
        print(f"   Available batches  : {sorted(df_schedule['batch_id'].unique())[:10]} ...")
        print(f"   Available machines : {sorted(df_schedule['machine_name'].unique())}")
        return

    # ── Title ────────────────────────────────────────────────────────────────
    if product_id != "All":
        t_product = f"Product {product_id} — {df.iloc[0]['product_desc']}"
    else:
        t_product = f"All Products ({df['product_item'].nunique()} products)"

    t_batches  = "All Batches"  if batch_ids   == "All" else f"Batches: {', '.join(batch_ids)}"
    t_machines = "All Machines" if machine_names == "All" else f"{len(machine_names)} machine(s)"

    # ── Y-axis order: machines by first appearance ───────────────────────────
    machines_order = list(dict.fromkeys(df.sort_values("start_time")["machine_name"]))
    machine_y      = {m: i for i, m in enumerate(machines_order)}

    # ── Colour key: batch (single product) or product (all) ─────────────────
    by_batch = (product_id != "All")
    keys     = sorted(df["batch_id"].unique()) if by_batch else sorted(df["product_item"].unique())
    legend_label = "Batch" if by_batch else "Product"

    palette = [
        "#AED6F1","#A9DFBF","#F9E79F","#F5CBA7","#D2B4DE",
        "#ABEBC6","#FAD7A0","#A8D8EA","#F1948A","#85C1E9",
        "#82E0AA","#F8C471","#BB8FCE","#76D7C4","#F0B27A",
    ]
    color_map = {k: palette[i % len(palette)] for i, k in enumerate(keys)}

    def bar_color(row):
        return color_map[row["batch_id"] if by_batch else row["product_item"]]

    # ── Figure ───────────────────────────────────────────────────────────────
    fig = go.Figure()

    for _, row in df.iterrows():
        y     = machine_y[row["machine_name"]]
        color = bar_color(row)
        dur_h = row["duration_hours"]
        lx    = row["start_time"] + (row["end_time"] - row["start_time"]) / 2

        # Bar rectangle
        fig.add_shape(
            type="rect",
            x0=row["start_time"], x1=row["end_time"],
            y0=y - 0.38,          y1=y + 0.38,
            fillcolor=color,
            line=dict(color="rgba(80,80,80,0.28)", width=1),
            layer="below",
        )

        # "Step N" label — show when bar is wide enough
        if dur_h >= 0.25:
            fig.add_annotation(
                x=lx, y=y + 0.10,
                text=f"<b>Step {row['step_number']}</b>",
                showarrow=False,
                font=dict(size=10, color="#1a252f"),
                align="center", xanchor="center",
            )
        # Batch ID sub-label
        if dur_h >= 0.55:
            fig.add_annotation(
                x=lx, y=y - 0.18,
                text=row["batch_id"],
                showarrow=False,
                font=dict(size=8, color="#555555"),
                align="center", xanchor="center",
            )

        # Hover (invisible scatter point)
        hover = (
            f"<b>Step {row['step_number']}: {row['step_name']}</b><br>"
            f"Batch:    {row['batch_id']}<br>"
            f"Product:  P{row['product_item']} — {row['product_desc'][:42]}<br>"
            f"Machine:  {row['machine_name']}<br>"
            f"Start:    {row['start_time'].strftime('%d %b %H:%M')}<br>"
            f"End:      {row['end_time'].strftime('%d %b %H:%M')}<br>"
            f"Duration: {dur_h:.2f}h"
        )
        fig.add_trace(go.Scatter(
            x=[lx], y=[y], mode="markers",
            marker=dict(size=1, opacity=0),
            hovertemplate=hover + "<extra></extra>",
            showlegend=False,
        ))

    # ── Legend entries ────────────────────────────────────────────────────────
    shown = set()
    for _, row in df.iterrows():
        k = row["batch_id"] if by_batch else row["product_item"]
        if k in shown: continue
        shown.add(k)
        if by_batch:
            legend_name = k
        else:
            desc = df[df["product_item"] == k].iloc[0]["product_desc"]
            legend_name = f"P{k}: {desc[:36]}{'...' if len(desc)>36 else ''}"
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=13, color=color_map[k], symbol="square",
                        line=dict(color="rgba(80,80,80,0.3)", width=1)),
            name=legend_name, showlegend=True,
        ))

    # ── Axis range & tick density ─────────────────────────────────────────────
    x_min  = df["start_time"].min()
    x_max  = df["end_time"].max()
    span_h = (x_max - x_min).total_seconds() / 3600
    dtick  = 3600000 if span_h <= 24 else 3600000 * 3    # 1h ticks ≤24h else 3h

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        title=dict(
            text=(
                f"<b>Production Timeline</b>  ·  {t_product}<br>"
                f"<span style='font-size:11px;color:#888'>"
                f"{t_batches}  ·  {t_machines}</span>"
            ),
            font=dict(size=14, color="#2c3e50"),
            x=0,
        ),
        xaxis=dict(
            title="",
            type="date",
            tickformat="%H:%M",
            dtick=dtick,
            showgrid=True,
            gridcolor="rgba(200,200,200,0.55)",
            gridwidth=1,
            zeroline=False,
            tickfont=dict(size=11, color="#555"),
            range=[x_min - timedelta(minutes=15), x_max + timedelta(minutes=15)],
            showline=True,
            linecolor="rgba(180,180,180,0.5)",
            ticks="outside", ticklen=4,
        ),
        yaxis=dict(
            title="",
            tickvals=list(range(len(machines_order))),
            ticktext=[f"<b>{m}</b>" for m in machines_order],
            showgrid=True,
            gridcolor="rgba(200,200,200,0.35)",
            zeroline=False,
            tickfont=dict(size=11, color="#333"),
            range=[-0.65, len(machines_order) - 0.35],
            autorange="reversed",      # first machine at top
        ),
        plot_bgcolor="#FAFAFA",
        paper_bgcolor="white",
        height=max(300, len(machines_order) * 85 + 140),
        width=1050,
        legend=dict(
            title=dict(text=legend_label, font=dict(size=11, color="#333")),
            orientation="v",
            x=1.01, y=1,
            bgcolor="white",
            bordercolor="rgba(0,0,0,0.1)",
            borderwidth=1,
            font=dict(size=10),
            itemsizing="constant",
        ),
        margin=dict(l=240, r=210, t=95, b=55),
        hovermode="closest",
    )

    fig.show()

    print(f"  ✅  Products: {df['product_item'].nunique()}  |  "
          f"Batches: {df['batch_id'].nunique()}  |  "
          f"Machines: {df['machine_name'].nunique()}  |  "
          f"Entries: {len(df)}")
    print(f"      Window: {x_min.strftime('%d %b %H:%M')} → {x_max.strftime('%d %b %H:%M')}")


print("✅ plot_gantt() defined — ready to use")
print()
print("Usage examples:")
print("  plot_gantt(df_schedule, product_id=1)")
print("  plot_gantt(df_schedule, product_id='All')")
print("  plot_gantt(df_schedule, product_id=2, batch_ids=['B-002-001'])")
print("  plot_gantt(df_schedule, product_id='All', machine_names=['Wire Cutting Station'])")


✅ plot_gantt() defined — ready to use

Usage examples:
  plot_gantt(df_schedule, product_id=1)
  plot_gantt(df_schedule, product_id='All')
  plot_gantt(df_schedule, product_id=2, batch_ids=['B-002-001'])
  plot_gantt(df_schedule, product_id='All', machine_names=['Wire Cutting Station'])


### 10a. Single Product — All Batches, All Machines


In [28]:
# ✏️ Change product_id to any number between 1 and NUM_PRODUCTS
plot_gantt(df_schedule, product_id=1)


  ✅  Products: 1  |  Batches: 3  |  Machines: 4  |  Entries: 18
      Window: 15 Jan 06:00 → 15 Jan 12:24


### 10b. All Products — Full Schedule

In [24]:
plot_gantt(df_schedule, product_id="All")


  ✅  Products: 12  |  Batches: 34  |  Machines: 8  |  Entries: 194
      Window: 15 Jan 06:00 → 16 Jan 10:15


### 10c. Custom Filter
✏️ Mix and match — any combination of product, batches, machines.


In [ ]:
# ═══════════════════════════════════════════════════════════
#  ✏️  EDIT FILTERS
# ═══════════════════════════════════════════════════════════

PRODUCT   = 3          # integer, or "All"

BATCHES   = ["B-001-001"]     # "All"  — or list like ["B-001-001", "B-001-002"]
# BATCHES = [b for b in df_schedule["batch_id"].unique() if b.startswith("B-001")]

MACHINES  = "All"      # "All"  — or list like ["Wire Cutting Station"]
# MACHINES = list(df_schedule["machine_name"].unique())   # all machines explicitly

# ═══════════════════════════════════════════════════════════
plot_gantt(df_schedule, product_id=PRODUCT, batch_ids=BATCHES, machine_names=MACHINES)


  ✅  Products: 1  |  Batches: 1  |  Machines: 4  |  Entries: 6
      Window: 15 Jan 06:00 → 15 Jan 09:49


### 10d. Helper — See Available Values

In [26]:
print("── Products ──────────────────────────────────────────")
for _, row in df_schedule.drop_duplicates("product_item").iterrows():
    print(f"  {row['product_item']:>3} | {row['product_desc']}")

print()
print("── Batches (first 15) ────────────────────────────────")
for b in sorted(df_schedule["batch_id"].unique())[:15]:
    pid = df_schedule[df_schedule["batch_id"]==b].iloc[0]["product_item"]
    print(f"  {b}  (Product {pid})")

print()
print("── Machines ──────────────────────────────────────────")
for m in sorted(df_schedule["machine_name"].unique()):
    n = len(df_schedule[df_schedule["machine_name"]==m])
    print(f"  {m}  ({n} steps scheduled)")


── Products ──────────────────────────────────────────
    1 | 4-Wire Jacket 1xMQS Module 16K915H
    2 | Coaxial Cable 1xHVL Module H22-105F
    3 | 3-Wire Jacket 2xFAKRA Module 365DG87
    4 | Single Wire 3xMini-Fit Module U20-640B
    5 | Shielded 4-Wire 1xHSD Module VQ29592A BA/BB/BC
    6 | 8-Wire Jacket 3xMini-Fit Module 84M305F
    7 | Twisted Quad 2xMini-Fit Module 14C821P DA/DB
    8 | Coaxial Cable 1xJPT Module 446-K20S
    9 | Single Wire 1xFAKRA Module 674MG43
   10 | Twisted Quad 1xHSD Module 63L878P BA/BB/BC
   11 | Twisted Pair 2xFAKRA Module O39114S
   12 | 4-Wire Jacket 2xRAST Module M43701J

── Batches (first 15) ────────────────────────────────
  B-001-001  (Product 1)
  B-001-002  (Product 1)
  B-001-003  (Product 1)
  B-002-001  (Product 2)
  B-002-002  (Product 2)
  B-002-003  (Product 2)
  B-003-001  (Product 3)
  B-003-002  (Product 3)
  B-003-003  (Product 3)
  B-004-001  (Product 4)
  B-004-002  (Product 4)
  B-004-003  (Product 4)
  B-005-001  (Product 5)
  B

## 11. Export Schedule

In [27]:
from datetime import datetime as dt
import json

df_export = df_schedule.copy()
df_export["start_time"] = df_export["start_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_export["end_time"]   = df_export["end_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_export.to_csv("production_schedule.csv", index=False)

with open("production_schedule.json", "w") as f:
    json.dump({
        "generated_at":  dt.now().isoformat(),
        "total_entries": len(df_export),
        "schedule":      df_export.to_dict(orient="records"),
    }, f, indent=2)

print("✅ Exported:")
print(f"   production_schedule.csv   ({len(df_export)} rows)")
print(f"   production_schedule.json  ({len(df_export)} entries)")
print()
print("Columns:", list(df_export.columns))


✅ Exported:
   production_schedule.csv   (194 rows)
   production_schedule.json  (194 entries)

Columns: ['batch_id', 'batch_num', 'batch_size', 'product_item', 'product_desc', 'step_number', 'step_name', 'machine_name', 'start_time', 'end_time', 'duration_hours']
